In [ ]:
import time
import math
import torch
import torch.nn.functional as F
from dataset import PAD_IDX, SOS_IDX, EOS_IDX

In [ ]:
def beam_search_decode(model, src_ids, src_mask, beam_size: int = 5, max_tgt_len: int = 128, length_penalty: float = 0.0, repetition_penalty: float = 1.0):
    device = src_ids.device
    k = beam_size

    encoder_out = model.encode(src_ids, src_mask)

    def expand_beam(t):
        if isinstance(t, torch.Tensor):
            if t.dim() == 3 and t.size(1) == 1:
                return t.expand(t.size(0), k, t.size(2)).contiguous()
            return t.expand(k, *t.shape[1:]).contiguous()
        elif isinstance(t, (tuple, list)):
            return type(t)([expand_beam(x) for x in t])
        return t

    encoder_out = expand_beam(encoder_out)
    src_mask = src_mask.expand(k, -1)
    hidden = model.init_decoder_hidden(encoder_out)

    token = torch.full((k,), SOS_IDX, dtype=torch.long, device=device)
    log_probs = torch.zeros(k, device=device)
    seqs = torch.full((k, 1), SOS_IDX, dtype=torch.long, device=device)
    finished = torch.zeros(k, dtype=torch.bool, device=device)
    completed = []

    for step in range(max_tgt_len):
        step_logits, hidden = model.decode_step(token, hidden, encoder_out, src_mask)
        vocab_size = step_logits.size(-1)

        if repetition_penalty != 1.0:
            for i in range(k):
                prev_tokens = seqs[i].unique()
                score = step_logits[i, prev_tokens]
                step_logits[i, prev_tokens] = torch.where(
                    score > 0, score / repetition_penalty, score * repetition_penalty
                )

        step_log_probs = F.log_softmax(step_logits, dim=-1)

        if finished.any():
            pad_mask = torch.full_like(step_log_probs, float("-inf"))
            pad_mask[:, PAD_IDX] = 0.0
            step_log_probs = torch.where(finished.unsqueeze(1), pad_mask, step_log_probs)

        candidate_scores = log_probs.unsqueeze(1) + step_log_probs

        if step == 0:
            candidate_scores = candidate_scores[0:1]

        flat = candidate_scores.reshape(-1)
        topk_scores, topk_idx = torch.topk(flat, k, dim=-1)

        beam_idx = topk_idx // vocab_size
        token_idx = topk_idx % vocab_size

        log_probs = topk_scores
        token = token_idx
        seqs = torch.cat([seqs[beam_idx], token_idx.unsqueeze(1)], dim=1)
        finished = finished[beam_idx] | (token_idx == EOS_IDX)

        hidden = _reorder_hidden(hidden, beam_idx)

        newly_finished = (token_idx == EOS_IDX)
        if newly_finished.any():
            for i in newly_finished.nonzero(as_tuple=False).squeeze(-1).tolist():
                seq = seqs[i, 1:]
                score = log_probs[i].item()
                if length_penalty > 0:
                    L = len(seq)
                    # Length Penalty Formula
                    penalty = ((5.0 + L) / 6.0) ** length_penalty
                    score /= penalty
                completed.append((score, seq))

        if finished.all():
            break

    for i in range(k):
        if not finished[i]:
            seq = seqs[i, 1:]
            score = log_probs[i].item()
            if length_penalty > 0:
                L = len(seq)
                penalty = ((5.0 + L) / 6.0) ** length_penalty
                score /= penalty
            completed.append((score, seq))

    completed.sort(key=lambda x: x[0], reverse=True)
    return completed[0][1]

In [ ]:
def _reorder_hidden(hidden, beam_idx):
    if isinstance(hidden, torch.Tensor):
        dim = 1 if hidden.dim() == 3 else 0
        return hidden.index_select(dim, beam_idx)
    elif isinstance(hidden, tuple):
        return tuple(_reorder_hidden(h, beam_idx) for h in hidden)
    elif isinstance(hidden, list):
        return [_reorder_hidden(h, beam_idx) for h in hidden]
    return hidden

In [ ]:
@torch.no_grad()
def decode_dataset(model, dataloader, beam_size: int = 5, max_tgt_len: int = 128, length_penalty: float = 0.0, repetition_penalty: float = 1.0, device: str = "cuda"):
    model.eval()
    model.to(device)

    predictions = []
    total_time = 0.0

    for batch in dataloader:
        src_ids = batch["src_ids"].to(device)
        src_mask = batch["src_attention_mask"].to(device)

        t0 = time.perf_counter()
        best_seq = beam_search_decode(
            model, src_ids, src_mask,
            beam_size=beam_size,
            max_tgt_len=max_tgt_len,
            length_penalty=length_penalty,
            repetition_penalty=repetition_penalty,
        )
        total_time += time.perf_counter() - t0

        predictions.append(best_seq.cpu())

    avg_time = total_time / max(len(predictions), 1)
    print(f"Decoded {len(predictions)} sentences | beam={beam_size} | avg {avg_time*1000:.1f} ms/sentence")

    return predictions, avg_time